# O*NET Skill Importance and Level EDA

This review uses Skill.zip only. One record is an occupation–skill pair. Code is retained as the occupation identifier for validation and future joins.

In [1]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
import pandas as pd

source = next(
    (folder for folder in (Path('data'), Path('../data')) if (folder / 'Skill.zip').is_file()),
    None,
)
if source is None:
    raise FileNotFoundError('Open this notebook from the project root or notebooks directory.')
attachments = source
with ZipFile(next(attachments.rglob('Skill.zip'))) as archive:
    eda = pd.concat([
        pd.read_excel(BytesIO(archive.read(name)), header=3,
                      dtype={'Code': str, 'Job Zone': str})
        .assign(Skill=Path(name).stem.replace('_', ' '))
        for name in archive.namelist() if name.endswith('.xlsx')
    ], ignore_index=True)
assert not eda.duplicated(['Code', 'Skill']).any()
eda['Level_observed'] = pd.to_numeric(eda.Level, errors='coerce')
eda['Level_status'] = eda.Level.where(eda.Level_observed.isna(), 'Rated')
assert set(eda.Level_status) == {'Rated', 'Not relevant', 'Not available'}
eda['Importance_band'] = pd.cut(
    eda.Importance, [-1, 19, 39, 59, 79, 100],
    labels=['0–19', '20–39', '40–59', '60–79', '80–100']
)
print(eda.Level_status.value_counts().to_string())


Level_status
Rated            10612
Not relevant      2050
Not available       78


## Estimate unavailable ratings

Only “Not available” is eligible for imputation. Calculate medians from observed ratings for the same skill, Job Zone and exact Importance. With fewer than five observed ratings, use the same skill and Job Zone within an Importance band. Preserve the raw value and flag each estimate. Leave “Not relevant” missing in both summaries.

In [2]:
def comparable_scores(importance_group):
    observed = eda.groupby(
        ['Skill', 'Job Zone', importance_group], observed=True
    ).Level_observed
    return observed.transform('median').where(observed.transform('count').ge(5))

unknown = eda.Level_status.eq('Not available')
exact = comparable_scores('Importance')
band = comparable_scores('Importance_band')
eda['Level_imputed'] = unknown & (exact.notna() | band.notna())
eda['Level_with_estimates'] = eda.Level_observed.fillna(exact.fillna(band).where(unknown))
eda['Imputation_method'] = 'None'
eda.loc[unknown & exact.notna(), 'Imputation_method'] = 'Exact Importance'
eda.loc[unknown & exact.isna() & band.notna(), 'Imputation_method'] = 'Importance band'
assert eda.loc[eda.Level_status.eq('Not relevant'), 'Level_with_estimates'].isna().all()
assert eda.Level_imputed.sum() == 78
print(eda.loc[unknown, 'Imputation_method'].value_counts().to_string())
print(eda.loc[unknown, ['Occupation', 'Skill', 'Level', 'Level_with_estimates',
                        'Imputation_method']].head(3).to_string(index=False))


Imputation_method
Exact Importance    60
Importance band     18
                           Occupation                   Skill         Level  Level_with_estimates Imputation_method
        Music Directors and Composers Complex Problem Solving Not available                  45.0  Exact Importance
Marine Engineers and Naval Architects   Equipment Maintenance Not available                  43.0   Importance band
       Geothermal Production Managers   Equipment Maintenance Not available                  27.0   Importance band


## Skill importance and level by preparation zone

Report the Level median under the selected policy, with separate counts of observed and estimated ratings.

In [3]:
summary = eda.groupby('Skill').agg(
    median_importance=('Importance', 'median'),
    median_level=('Level_with_estimates', 'median'),
    observed_level_n=('Level_observed', 'count'),
    estimated_level_n=('Level_imputed', 'sum'),
)
print(summary.to_string())

zone_summary = eda.groupby(['Skill', 'Job Zone']).agg(
    median_importance=('Importance', 'median'),
    median_level=('Level_with_estimates', 'median'),
    numeric_level_n=('Level_with_estimates', 'count'),
    estimated_level_n=('Level_imputed', 'sum'),
)
print(zone_summary.to_string())


                                   median_importance  median_level  observed_level_n  estimated_level_n
Skill                                                                                                  
Complex Problem Solving                         53.0          43.0               909                  1
Coordination                                    50.0          43.0               910                  0
Equipment Maintenance                           10.0          29.0               451                  5
Equipment Selection                             16.0          27.0               519                  9
Installation                                     0.0          21.0               163                 11
Instructing                                     47.0          43.0               906                  0
Judgment and Decision Making                    53.0          45.0               909                  1
Management of Financial Resources               19.0          16

## Skill importance ranks within the four clusters

Rank the 14 skills by mean Importance within each cluster. Reference Level is the median among numeric Level ratings under the cleaning policy above. The HTML uses the top five skills in the selected or matched group for an optional importance-weighted keyword coverage check. Each matched skill contributes its mean Importance divided by the sum of the five means. Reference Level describes the occupations in the group.

In [4]:
assignments = pd.read_csv(source / 'occupation_cluster_assignments.csv',
                          dtype={'occupation_code': str})
cluster_skills = eda.merge(
    assignments[['occupation_code', 'cluster', 'cluster_name']],
    left_on='Code', right_on='occupation_code', validate='many_to_one',
)
assert len(cluster_skills) == len(eda)
cluster_skill_rankings = cluster_skills.groupby(['cluster', 'cluster_name', 'Skill']).agg(
    mean_importance=('Importance', 'mean'),
    median_reference_level=('Level_with_estimates', 'median'),
    level_ratings=('Level_with_estimates', 'count'),
    level_estimates=('Level_imputed', 'sum'),
).reset_index().sort_values(['cluster', 'mean_importance', 'Skill'],
                           ascending=[True, False, True])
cluster_skill_rankings['rank'] = cluster_skill_rankings.groupby('cluster').cumcount() + 1
print(cluster_skill_rankings.loc[cluster_skill_rankings['rank'].le(5),
      ['cluster', 'rank', 'Skill', 'mean_importance', 'median_reference_level', 'level_ratings']]
      .round(1).to_string(index=False))


 cluster  rank                             Skill  mean_importance  median_reference_level  level_ratings
       1     1      Judgment and Decision Making             68.0                    55.0            128
       1     2                      Coordination             67.2                    55.0            128
       1     3           Complex Problem Solving             66.1                    54.5            128
       1     4 Management of Personnel Resources             60.0                    54.0            128
       1     5                        Persuasion             58.1                    52.0            128
       2     1                      Coordination             46.0                    41.0            214
       2     2      Judgment and Decision Making             45.0                    36.0            214
       2     3           Complex Problem Solving             41.5                    35.0            214
       2     4                        Persuasion       